# Week 6 Capstone — Hand-coding the Engine of Deep Learning

整個暑期你都在替這一刻攢零件。今天用**純 numpy、從零**手刻三個會「學習」的模型,
過程中會一再看到同一條微積分骨架:

> **forward = function composition** &nbsp;→&nbsp; **loss** &nbsp;→&nbsp; **backward = chain rule (backprop)** &nbsp;→&nbsp; **update = gradient descent**

三個關卡,難度層層加深:
1. **Linear regression** — 梯度下降擬合一條有雜訊的直線,畫出 loss 曲線。
2. **Logistic regression** — sigmoid + cross-entropy,做 2D 二元分類,畫 decision boundary。
3. **2-layer neural network** — 一個隱藏層 + backpropagation,把鏈鎖法則自動化,學一組**非線性可分**的同心圓資料。

**Usage**: 上傳到 [Google Colab](https://colab.research.google.com/) 或本機 Jupyter,由上往下逐格執行。
每個模型都是純 numpy、不用任何 ML 套件——因為引擎的核心就是微積分,我們要親手刻出來。

> 圖表標籤一律用英文/數學符號,避免中文變「豆腐字」;中文說明都放在文字格。

In [ ]:
# === 環境設定(先跑這格)===
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (6.5, 4)
plt.rcParams['axes.grid'] = True

def sigmoid(z):
    # 數值穩定的 sigmoid:把 z 夾住避免 exp 溢位
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

print("環境就緒, numpy", np.__version__)

### (選用)讓圖表顯示中文

預設圖表用英文標籤,避免中文變「豆腐字」。若你在 Colab 想要中文座標/標題,
把下一格的註解取消再執行(只需一次),之後的圖就能顯示中文。

In [ ]:
# 想要中文圖標時,取消以下註解執行(Colab 適用;本機 Jupyter 需自備 CJK 字型)
# !apt-get -qq install fonts-noto-cjk > /dev/null
# import matplotlib
# matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# matplotlib.rcParams['axes.unicode_minus'] = False
print("預設英文標籤;要中文請見上一格說明")

## Part 1｜Linear Regression by Gradient Descent

**目標**:給一堆帶雜訊的點 $(x_i, y_i)$,找一條直線 $\hat y = wx+b$ 把它們擬合起來。

微積分骨架第一次登場:

| 步驟 | 這裡是什麼 |
|---|---|
| **forward** | $\hat y = wx+b$(一個簡單的函數) |
| **loss** | 均方誤差 $L=\frac1N\sum(\hat y_i-y_i)^2$ |
| **backward** | 對 $w,b$ 偏微分求梯度 $\dfrac{\partial L}{\partial w},\dfrac{\partial L}{\partial b}$ |
| **update** | $w \leftarrow w-\eta\,\dfrac{\partial L}{\partial w}$(往下坡走一步) |

先自己生一組玩具資料:真直線 $y=2x-1$ 加上高斯雜訊。

In [ ]:
# 自生 toy dataset:真直線 y = 2x - 1 + 雜訊
rng = np.random.default_rng(0)
N = 100
X = np.linspace(-3, 3, N)
true_w, true_b = 2.0, -1.0
y = true_w * X + true_b + rng.normal(0, 0.8, N)

plt.scatter(X, y, s=12, label='data')
plt.plot(X, true_w * X + true_b, 'k--', label='true line y = 2x - 1')
plt.title('Part 1: noisy linear data'); plt.legend(); plt.show()

In [ ]:
# 手刻梯度下降:forward -> loss -> backward -> update
w, b = 0.0, 0.0          # 從零開始
lr = 0.03                # learning rate (步長)
losses = []
for step in range(600):
    y_hat = w * X + b                 # forward:線性函數
    err = y_hat - y
    loss = np.mean(err**2)            # loss:MSE
    losses.append(loss)
    # backward:L 對 w、b 的偏導(這就是微積分——鏈鎖 + 冪次法則)
    dw = 2 * np.mean(err * X)
    db = 2 * np.mean(err)
    # update:沿負梯度走一步(gradient descent)
    w -= lr * dw
    b -= lr * db

print(f"learned:  w = {w:.4f}  (true 2.0),   b = {b:.4f}  (true -1.0)")
print(f"final MSE loss = {losses[-1]:.4f}")

In [ ]:
# 看結果:擬合線 + loss 曲線
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(X, y, s=12, label='data')
ax[0].plot(X, w * X + b, 'r', lw=2, label=f'fit: y = {w:.2f}x + {b:.2f}')
ax[0].set_title('Part 1: fitted line'); ax[0].legend()
ax[1].plot(losses); ax[1].set_xlabel('gradient-descent step'); ax[1].set_ylabel('MSE loss')
ax[1].set_title('Part 1: training loss goes down')
plt.tight_layout(); plt.show()

## Part 2｜Logistic Regression (2D binary classification)

現在資料有**兩類**(標籤 $0$ / $1$),要學一條分界線。把線性輸出 $z=w\cdot x+b$
經過 **sigmoid** $\sigma(z)=\dfrac{1}{1+e^{-z}}$ 壓成機率 $p\in(0,1)$。

- **forward**: $z = w\cdot x + b$,$p=\sigma(z)$
- **loss**: cross-entropy $L=-\frac1N\sum\big[y\ln p+(1-y)\ln(1-p)\big]$
- **backward**: 神奇的是,鏈鎖法則一路化簡後,梯度就是 $\dfrac{\partial L}{\partial z}=p-y$(乾淨到不可思議)
- **update**: 一樣是梯度下降

先自生兩坨高斯點當兩個類別。

In [ ]:
# 自生 toy dataset:兩坨高斯點(線性可分)
rng = np.random.default_rng(1)
n = 150
X0 = rng.normal(0, 0.8, (n, 2)) + np.array([-1.6, -1.6])   # class 0
X1 = rng.normal(0, 0.8, (n, 2)) + np.array([ 1.6,  1.6])   # class 1
Xl = np.vstack([X0, X1])
yl = np.concatenate([np.zeros(n), np.ones(n)])

plt.scatter(Xl[:, 0], Xl[:, 1], c=yl, cmap='bwr', edgecolors='k', s=14)
plt.title('Part 2: two classes'); plt.xlabel('x1'); plt.ylabel('x2'); plt.show()

In [ ]:
# 手刻邏輯回歸:sigmoid + cross-entropy + 梯度下降
w = np.zeros(2); b = 0.0
lr = 0.1
ce = []
for step in range(3000):
    z = Xl @ w + b                   # forward:線性
    p = sigmoid(z)                   # forward:sigmoid -> 機率
    loss = -np.mean(yl*np.log(p+1e-9) + (1-yl)*np.log(1-p+1e-9))   # cross-entropy
    ce.append(loss)
    # backward:鏈鎖法則化簡後,dL/dz = p - y
    grad_w = Xl.T @ (p - yl) / len(yl)
    grad_b = np.mean(p - yl)
    w -= lr * grad_w; b -= lr * grad_b   # update:gradient descent

pred = (sigmoid(Xl @ w + b) > 0.5).astype(float)
acc = np.mean(pred == yl)
print(f"weights w = {w},  bias b = {b:.4f}")
print(f"final cross-entropy = {ce[-1]:.4f},   accuracy = {acc:.4f}")

In [ ]:
# 畫出 decision boundary(sigmoid = 0.5 的那條線)
xx, yy = np.meshgrid(np.linspace(-4, 4, 300), np.linspace(-4, 4, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = sigmoid(grid @ w + b).reshape(xx.shape)
plt.contourf(xx, yy, zz, levels=20, cmap='bwr', alpha=0.6)
plt.contour(xx, yy, zz, levels=[0.5], colors='k', linewidths=2)
plt.scatter(Xl[:, 0], Xl[:, 1], c=yl, cmap='bwr', edgecolors='k', s=14)
plt.title('Part 2: logistic-regression decision boundary')
plt.xlabel('x1'); plt.ylabel('x2'); plt.show()

## Part 3｜A 2-Layer Neural Network + Backpropagation

邏輯回歸只會畫**直線**。但真實資料常常非線性可分——例如**同心圓**:內圈一類、外圈一類,
沒有任何直線能分開它們。這時需要在中間加一個**隱藏層**,讓模型能彎折。

網路結構(input 2 → hidden $H$ 個 tanh 神經元 → output 1 個 sigmoid):

$$
\underbrace{Z_1 = XW_1+b_1,\quad A_1=\tanh(Z_1)}_{\text{hidden layer}}
\;\longrightarrow\;
\underbrace{Z_2 = A_1W_2+b_2,\quad P=\sigma(Z_2)}_{\text{output}}
$$

**forward 就是一連串函數合成**。要更新參數就得算 $\dfrac{\partial L}{\partial W_1},\dots$——
一層層往回套**鏈鎖法則**,這正是 **backpropagation**:

$$
\frac{\partial L}{\partial Z_1}
=\underbrace{\frac{\partial L}{\partial Z_2}}_{P-Y}\;
\underbrace{\frac{\partial Z_2}{\partial A_1}}_{W_2^\top}\;
\underbrace{\frac{\partial A_1}{\partial Z_1}}_{1-\tanh^2}
$$

backprop = 把鏈鎖法則**大規模自動化**。先自生同心圓資料。

In [ ]:
# 自生 toy dataset:同心圓(非線性可分)——內圈 label 0、外圈 label 1
rng = np.random.default_rng(2)
m = 400
angle = rng.uniform(0, 2*np.pi, m)
radius = np.concatenate([rng.uniform(0.0, 1.0, m//2),      # 內圈
                         rng.uniform(1.8, 2.8, m//2)])     # 外圈
Xn = np.stack([radius*np.cos(angle), radius*np.sin(angle)], axis=1)
Xn += rng.normal(0, 0.06, Xn.shape)
yn = np.concatenate([np.zeros(m//2), np.ones(m//2)]).reshape(-1, 1)

plt.scatter(Xn[:, 0], Xn[:, 1], c=yn.ravel(), cmap='bwr', edgecolors='k', s=14)
plt.gca().set_aspect('equal'); plt.title('Part 3: concentric circles (not linearly separable)')
plt.xlabel('x1'); plt.ylabel('x2'); plt.show()

In [ ]:
# 初始化參數(小隨機權重打破對稱)
H = 16                                  # 隱藏層神經元數
rng = np.random.default_rng(2)
W1 = rng.normal(0, 1, (2, H)) * 0.8; b1 = np.zeros(H)
W2 = rng.normal(0, 1, (H, 1)) * 0.8; b2 = np.zeros(1)
print("params:  W1", W1.shape, " W2", W2.shape)

In [ ]:
# 手刻訓練迴圈:forward(合成)-> loss -> backward(鏈鎖/backprop)-> update(GD)
lr = 0.2
nn_loss = []
for epoch in range(4000):
    # ---- forward:一連串函數合成 ----
    Z1 = Xn @ W1 + b1
    A1 = np.tanh(Z1)                    # 隱藏層 activation
    Z2 = A1 @ W2 + b2
    P  = sigmoid(Z2)                    # 輸出機率
    loss = -np.mean(yn*np.log(P+1e-9) + (1-yn)*np.log(1-P+1e-9))
    nn_loss.append(loss)
    # ---- backward:鏈鎖法則一層層往回(backpropagation)----
    dZ2 = (P - yn) / len(yn)            # dL/dZ2
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(0)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (1 - A1**2)            # 乘上 tanh 的導數 1 - tanh^2
    dW1 = Xn.T @ dZ1
    db1 = dZ1.sum(0)
    # ---- update:gradient descent ----
    W1 -= lr*dW1; b1 -= lr*db1; W2 -= lr*dW2; b2 -= lr*db2

def nn_forward(A):
    return sigmoid(np.tanh(A @ W1 + b1) @ W2 + b2)

acc = np.mean((nn_forward(Xn) > 0.5).astype(float) == yn)
print(f"final cross-entropy = {nn_loss[-1]:.4f},   accuracy = {acc:.4f}")

In [ ]:
# 畫出 NN 學到的(彎曲的!)decision boundary + loss 曲線
xx, yy = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(-3, 3, 300))
zz = nn_forward(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].contourf(xx, yy, zz, levels=20, cmap='bwr', alpha=0.6)
ax[0].contour(xx, yy, zz, levels=[0.5], colors='k', linewidths=2)
ax[0].scatter(Xn[:, 0], Xn[:, 1], c=yn.ravel(), cmap='bwr', edgecolors='k', s=12)
ax[0].set_aspect('equal'); ax[0].set_title('Part 3: NN decision boundary (curved!)')
ax[0].set_xlabel('x1'); ax[0].set_ylabel('x2')
ax[1].plot(nn_loss); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('cross-entropy loss')
ax[1].set_title('Part 3: training loss')
plt.tight_layout(); plt.show()

## 收尾 · 你剛手刻了深度學習引擎的核心

停下來看看你剛剛做了什麼。三個模型、從線性到非線性,骨架**完全一樣**:

| 步驟 | 微積分是什麼 | 三個模型共通 |
|---|---|---|
| **forward** | 函數**合成** | 把輸入一層層代進函數 |
| **loss** | 一個要最小化的純量函數 | MSE / cross-entropy |
| **backward** | **鏈鎖法則** | 從 loss 往回逐層求偏導 = **backprop** |
| **update** | 沿**負梯度**下降 | $\theta \leftarrow \theta-\eta\nabla_\theta L$ |

那個 2 層神經網路的反向傳播,骨子裡就是你在第 2 週學的**鏈鎖法則**——
只是被排成矩陣、大規模自動化了。把 $H$ 加大、層數疊深,再換個名字,
它就叫 **deep learning**。工業界的 PyTorch / TensorFlow 幫你自動算的 `.backward()`,
跟你這裡手寫的 `dZ1 = dA1 * (1 - A1**2)` 是同一件事。

> **你剛手刻了深度學習引擎的核心。** 微積分不是期中考完就丟的東西——
> 它就是讓機器「學習」的那台引擎。整個暑期的極限、導數、鏈鎖法則、最佳化,
> 在這一頁全部合體了。

### 進階徽章(選做)
1. 把 Part 3 的同心圓換成 **XOR**(`yn = ((Xn[:,0] > 0) ^ (Xn[:,1] > 0))`),重新訓練,看 boundary 變成什麼。
2. 把隱藏層神經元 `H` 從 16 調到 2,再調到 64,觀察 boundary 與最終 accuracy 怎麼變。
3. 幫 Part 1 的梯度下降加上「每 100 步印一次 loss」,親眼看它單調下降。
4. 把 Part 3 的 `tanh` 換成 `relu`(記得 relu 的導數是 `(Z1 > 0)`),比較收斂速度。